# 02 — Data Preprocessing

Load PV-equipped building CSVs, normalize PV by installed capacity, merge with weather, engineer features, and save train/test splits.

In [15]:
import os
import numpy as np
import pandas as pd
from pathlib import Path

In [16]:
# ── Paths ──
BASE_DIR = Path(os.getcwd()).parent
RAW_DIR = BASE_DIR / "data" / "HEEW-20250912" / "raw data"
OUT_DIR = BASE_DIR / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

WEATHER_FILE = RAW_DIR / "Total_weather.csv"
print(f"Raw data dir: {RAW_DIR}")
print(f"Output dir:   {OUT_DIR}")

Raw data dir: /home/gujjavarshith/Documents/SEM 6/FSD 2/Solar_energy_prediction/data/HEEW-20250912/raw data
Output dir:   /home/gujjavarshith/Documents/SEM 6/FSD 2/Solar_energy_prediction/data/processed


## 1. Load & Filter PV Buildings

Only keep buildings where `PV.max() > 0` (i.e., they have solar panels installed).

In [17]:
building_frames = []

for fname in sorted(os.listdir(RAW_DIR)):
    if not fname.endswith("_energy.csv") or fname.startswith("Total"):
        continue

    bld_id = fname.replace("_energy.csv", "")
    df = pd.read_csv(RAW_DIR / fname)

    # Drop NaN rows (leading blocks before monitoring started)
    df = df.dropna(subset=["PV"]).reset_index(drop=True)
    if len(df) == 0:
        continue

    pv_max = df["PV"].max()
    if pv_max <= 0:
        continue  # No PV system installed

    # Compute installed capacity and normalize PV
    installed_capacity = pv_max / 0.8
    df["PV_normalized"] = (df["PV"] / installed_capacity).clip(0, 1)
    df["installed_capacity"] = installed_capacity
    df["building_id"] = bld_id

    # Drop energy columns not needed as features
    df = df.drop(columns=["Electricity", "Cooling", "Heat",
                           "Total Energy", "Emission", "PV"])

    building_frames.append(df)
    print(f"{bld_id}: {len(df):,} rows, PV_max={pv_max:.2f}, cap={installed_capacity:.2f}")

print(f"\n{len(building_frames)} buildings with PV systems")

energy_df = pd.concat(building_frames, ignore_index=True)
print(f"Total rows: {len(energy_df):,}")

BN002: 78,868 rows, PV_max=68.34, cap=85.42
BN032: 78,870 rows, PV_max=74.50, cap=93.12
BN037: 78,870 rows, PV_max=22.67, cap=28.34
BN040: 78,870 rows, PV_max=134.73, cap=168.41
BN041: 78,870 rows, PV_max=73.69, cap=92.11
BN042: 78,870 rows, PV_max=61.44, cap=76.80
BN045: 78,867 rows, PV_max=179.55, cap=224.44
BN047: 78,830 rows, PV_max=61.40, cap=76.75
BN050: 78,870 rows, PV_max=17.15, cap=21.44
BN052: 78,870 rows, PV_max=49.77, cap=62.21
BN054: 78,866 rows, PV_max=49.99, cap=62.49
BN055: 78,870 rows, PV_max=14.51, cap=18.14
BN056: 78,870 rows, PV_max=50.04, cap=62.55
BN057: 78,868 rows, PV_max=49.22, cap=61.52
BN063: 78,868 rows, PV_max=83.24, cap=104.05
BN064: 78,843 rows, PV_max=200.53, cap=250.66
BN066: 78,865 rows, PV_max=324.80, cap=406.00
BN069: 78,868 rows, PV_max=100.58, cap=125.72
BN070: 78,868 rows, PV_max=67.87, cap=84.84
BN071: 78,868 rows, PV_max=28.72, cap=35.90
BN072: 78,869 rows, PV_max=214.74, cap=268.43
BN074: 78,868 rows, PV_max=50.29, cap=62.86
BN075: 78,870 rows,

## 2. Load Weather Data

In [18]:
weather_df = pd.read_csv(WEATHER_FILE)
weather_df = weather_df.rename(columns={"DOW": "Weekday"})
weather_df = weather_df.dropna(subset=["Temperature"]).reset_index(drop=True)
print(f"Weather shape: {weather_df.shape}")
weather_df.head()

Weather shape: (78701, 14)


,Year,Month,Day,Hour,Weekday,Temperature,Dew Point,Humidity,Wind,Wind Speed,Wind Gust,Pressure,Precip,Condition
0,2014,1,1,0,2,53.0,0.0,45.0,CALM,0.0,0.0,28.86,0.0,Fair
1,2014,1,1,1,2,51.0,33.0,50.0,E,5.0,0.0,28.86,0.0,Fair
2,2014,1,1,2,2,50.0,0.0,50.0,ESE,3.0,0.0,28.86,0.0,Fair
3,2014,1,1,3,2,49.0,0.0,52.0,ESE,6.0,0.0,28.86,0.0,Fair
4,2014,1,1,4,2,49.0,0.0,52.0,CALM,0.0,0.0,28.87,0.0,Fair


## 3. Merge Energy + Weather

In [19]:
merge_keys = ["Year", "Month", "Day", "Hour"]
weather_for_merge = weather_df.drop(columns=["Weekday"])  # energy already has it

merged = pd.merge(energy_df, weather_for_merge, on=merge_keys, how="inner")
print(f"Merged shape: {merged.shape}")
merged.head()

Merged shape: (3922883, 17)


,Year,Month,Day,Hour,Weekday,PV_normalized,installed_capacity,building_id,Temperature,Dew Point,Humidity,Wind,Wind Speed,Wind Gust,Pressure,Precip,Condition
0,2014,1,1,0,2,0.0,85.425,BN002,53.0,0.0,45.0,CALM,0.0,0.0,28.86,0.0,Fair
1,2014,1,1,1,2,0.0,85.425,BN002,51.0,33.0,50.0,E,5.0,0.0,28.86,0.0,Fair
2,2014,1,1,2,2,0.0,85.425,BN002,50.0,0.0,50.0,ESE,3.0,0.0,28.86,0.0,Fair
3,2014,1,1,3,2,0.0,85.425,BN002,49.0,0.0,52.0,ESE,6.0,0.0,28.86,0.0,Fair
4,2014,1,1,4,2,0.0,85.425,BN002,49.0,0.0,52.0,CALM,0.0,0.0,28.87,0.0,Fair


## 4. Feature Engineering

### 4a. Cyclical Time Encoding

In [20]:
# Sort by building + time first
merged = merged.sort_values(
    ["building_id", "Year", "Month", "Day", "Hour"]
).reset_index(drop=True)

# Cyclical encoding
merged["Hour_sin"]    = np.sin(2 * np.pi * merged["Hour"] / 24)
merged["Hour_cos"]    = np.cos(2 * np.pi * merged["Hour"] / 24)
merged["Month_sin"]   = np.sin(2 * np.pi * merged["Month"] / 12)
merged["Month_cos"]   = np.cos(2 * np.pi * merged["Month"] / 12)
merged["Weekday_sin"] = np.sin(2 * np.pi * merged["Weekday"] / 7)
merged["Weekday_cos"] = np.cos(2 * np.pi * merged["Weekday"] / 7)

print("Cyclical features added")

Cyclical features added


### 4b. One-Hot Encode Categoricals (Wind, Condition)

In [21]:
merged["Wind"] = merged["Wind"].fillna("UNKNOWN")
merged["Condition"] = merged["Condition"].fillna("UNKNOWN")

wind_dummies = pd.get_dummies(merged["Wind"], prefix="Wind", dtype=int)
cond_dummies = pd.get_dummies(merged["Condition"], prefix="Cond", dtype=int)
merged = pd.concat([merged, wind_dummies, cond_dummies], axis=1)

print(f"One-hot: {wind_dummies.shape[1]} Wind cols + {cond_dummies.shape[1]} Condition cols")

One-hot: 19 Wind cols + 18 Condition cols


### 4c. Lag & Rolling Features (per building)

In [22]:
grp = merged.groupby("building_id")["PV_normalized"]

merged["PV_norm_lag1"]   = grp.shift(1)
merged["PV_norm_lag24"]  = grp.shift(24)
merged["PV_norm_roll24"] = grp.transform(lambda x: x.rolling(24, min_periods=1).mean())

print("Lag & rolling features added")

Lag & rolling features added


### 4d. Label Encode building_id

In [23]:
bld_map = {b: i for i, b in enumerate(sorted(merged["building_id"].unique()))}
print(f"Building map: {bld_map}")
merged["building_id"] = merged["building_id"].map(bld_map)

Building map: {'BN002': 0, 'BN032': 1, 'BN037': 2, 'BN040': 3, 'BN041': 4, 'BN042': 5, 'BN045': 6, 'BN047': 7, 'BN050': 8, 'BN052': 9, 'BN054': 10, 'BN055': 11, 'BN056': 12, 'BN057': 13, 'BN063': 14, 'BN064': 15, 'BN066': 16, 'BN069': 17, 'BN070': 18, 'BN071': 19, 'BN072': 20, 'BN074': 21, 'BN075': 22, 'BN076': 23, 'BN079': 24, 'BN082': 25, 'BN089': 26, 'BN091': 27, 'BN094': 28, 'BN097': 29, 'BN098': 30, 'BN099': 31, 'BN100': 32, 'BN103': 33, 'BN105': 34, 'BN114': 35, 'BN115': 36, 'BN119': 37, 'BN126': 38, 'BN127': 39, 'BN129': 40, 'BN130': 41, 'BN131': 42, 'BN132': 43, 'BN136': 44, 'BN144': 45, 'CN01': 46, 'CN02': 47, 'CN03': 48, 'CN04': 49}


## 5. Train / Test Split (Temporal)

Split on Year **before** dropping it: train = 2014–2020, test = 2021–2022.

In [24]:
train_df = merged[merged["Year"] <= 2020].copy()
test_df  = merged[merged["Year"] >= 2021].copy()

print(f"Train (≤2020): {train_df.shape}")
print(f"Test  (≥2021): {test_df.shape}")

Train (≤2020): (3049034, 63)
Test  (≥2021): (873849, 63)


## 6. Drop Raw Date & Categorical Columns

Now that splitting is done, drop Year/Month/Day/Hour/Weekday + raw Wind/Condition.

In [25]:
drop_cols = ["Year", "Month", "Day", "Hour", "Weekday", "Wind", "Condition"]
train_df = train_df.drop(columns=drop_cols)
test_df  = test_df.drop(columns=drop_cols)

# Drop NaN rows from lag features (first 24 rows per building)
before_train, before_test = len(train_df), len(test_df)
train_df = train_df.dropna().reset_index(drop=True)
test_df  = test_df.dropna().reset_index(drop=True)

print(f"Dropped {before_train - len(train_df)} NaN rows from train (lag warmup)")
print(f"Dropped {before_test - len(test_df)} NaN rows from test")
print(f"\nTrain final: {train_df.shape}")
print(f"Test final:  {test_df.shape}")

Dropped 1200 NaN rows from train (lag warmup)
Dropped 0 NaN rows from test

Train final: (3047834, 56)
Test final:  (873849, 56)


In [26]:
# Preview the final columns
print(f"Columns ({len(train_df.columns)}):\n")
for i, col in enumerate(train_df.columns):
    print(f"  {i+1:2d}. {col}")

Columns (56):

   1. PV_normalized
   2. installed_capacity
   3. building_id
   4. Temperature
   5. Dew Point
   6. Humidity
   7. Wind Speed
   8. Wind Gust
   9. Pressure
  10. Precip
  11. Hour_sin
  12. Hour_cos
  13. Month_sin
  14. Month_cos
  15. Weekday_sin
  16. Weekday_cos
  17. Wind_CALM
  18. Wind_E
  19. Wind_ENE
  20. Wind_ESE
  21. Wind_N
  22. Wind_NE
  23. Wind_NNE
  24. Wind_NNW
  25. Wind_NW
  26. Wind_S
  27. Wind_SE
  28. Wind_SSE
  29. Wind_SSW
  30. Wind_SW
  31. Wind_UNKNOWN
  32. Wind_VAR
  33. Wind_W
  34. Wind_WNW
  35. Wind_WSW
  36. Cond_Blowing
  37. Cond_Cloudy
  38. Cond_Duststorm
  39. Cond_Fair
  40. Cond_Fog
  41. Cond_Haze
  42. Cond_Heavy
  43. Cond_Light
  44. Cond_Mostly
  45. Cond_Partly
  46. Cond_Patches
  47. Cond_Rain
  48. Cond_Smoke
  49. Cond_Squalls
  50. Cond_T-Storm
  51. Cond_Thunder
  52. Cond_UNKNOWN
  53. Cond_Widespread
  54. PV_norm_lag1
  55. PV_norm_lag24
  56. PV_norm_roll24


In [27]:
train_df.head()

,PV_normalized,installed_capacity,building_id,Temperature,Dew Point,Humidity,Wind Speed,Wind Gust,Pressure,Precip,...,Cond_Rain,Cond_Smoke,Cond_Squalls,Cond_T-Storm,Cond_Thunder,Cond_UNKNOWN,Cond_Widespread,PV_norm_lag1,PV_norm_lag24,PV_norm_roll24
0,0.0,85.425,0,51.0,35.0,54.0,0.0,0.0,28.89,0.0,...,0,0,0,0,0,0,0,0.0,0.0,0.139245
1,0.0,85.425,0,51.0,36.0,56.0,0.0,0.0,28.89,0.0,...,0,0,0,0,0,0,0,0.0,0.0,0.139245
2,0.0,85.425,0,50.0,36.0,59.0,3.0,0.0,28.89,0.0,...,0,0,0,0,0,0,0,0.0,0.0,0.139245
3,0.0,85.425,0,48.0,35.0,61.0,0.0,0.0,28.90,0.0,...,0,0,0,0,0,0,0,0.0,0.0,0.139245
4,0.0,85.425,0,50.0,33.0,52.0,5.0,0.0,28.90,0.0,...,0,0,0,0,0,0,0,0.0,0.0,0.139245


## 7. Save Processed Data

In [28]:
train_df.to_csv(OUT_DIR / "train.csv", index=False)
test_df.to_csv(OUT_DIR / "test.csv", index=False)

print(f"Saved {OUT_DIR / 'train.csv'} ({train_df.shape})")
print(f"Saved {OUT_DIR / 'test.csv'} ({test_df.shape})")

Saved /home/gujjavarshith/Documents/SEM 6/FSD 2/Solar_energy_prediction/data/processed/train.csv ((3047834, 56))
Saved /home/gujjavarshith/Documents/SEM 6/FSD 2/Solar_energy_prediction/data/processed/test.csv ((873849, 56))
